# LightGBM 信贷违约风控建模项目
**基于 Home Credit application_train · 完整建模流程 · 暂不校准**

参考你的 `catboost最终版.ipynb` 和 `逻辑回归.ipynb`：保留易于逐段运行的函数与分轮搜索结构，
融合LR中的时间/比例特征加工，采用CatBoost稿中的 **70% Train / 15% Selection / 15% Test**、
`random_state=42`、Early Stopping、AR容忍度与泛化gap选模。

**怎么运行：**把CSV放在本文件旁边，或在配置单元填写 `CSV_PATH`，然后从上往下运行。
默认 `RUN_MODE='standard'` 使用全部样本与完整搜索；先检查环境可改为 `quick`（仍用全量数据）。
`smoke` 只用于程序测试，不能把其结果当作正式效果。无需运行其他notebook。

**范围边界：**输出是未经校准的模型风险概率 `raw_probability`，越大风险越高；不生成PD校准、
评级映射或自动拒贷规则。这个公开表没有可靠的业务申请日期和未来观察窗元数据，随机Test不是OOT。
投产还需核验标签成熟期、贷前可得性、客户去重/时间外验证、合规与公平性、业务审批和监控阈值。

### 流程与参考稿的对应

| 环节 | 本文件的处理 |
|---|---|
| 数据划分 | 与参考稿同样的两次分层切分；不重排/删除全量记录 |
| 特征工程 | 修正365243；采用LR的5个年限变量和6个比例变量 |
| 筛查 | 仅在Train做常量/近常量/ID型筛查，记录剔除原因 |
| 模型输入 | 原生缺失值 + pandas类别变量；类别表仅由Train确定 |
| 参数搜索 | Baseline → 复杂度 → 学习率与正则 → 采样与正则；所有轮次一起选最优 |
| 最终训练 | 固定参数和树数，仅用Train重训，之后才打开Test |
| 交付与复用 | AUC/AR/KS/AP、PSI、Lift、SHAP、模型/规则保存、独立评分脚本与回读验证 |

与原CatBoost稿相比，本文件增加了特征工程并默认排除 `CODE_GENDER`；**相同样本划分不等于相同特征集**。
公平比较算法时，请让所有模型采用同样的输入政策与特征工程。这里不会把不同特征集的差异全归因于算法。

### 交付前验证范围

已在IPython中按单元格顺序实际执行：**全量307,511行的quick模式**（7组候选、每组最多500轮），
以及12,000行的完整搜索网格测试（每组最多80轮）。模型重训、最终评估、PSI、SHAP、文件保存与回读均通过。
另外验证了独立进程评分、列顺序变化、未知类别、数值缺失、非法数值、配套文件错用和PSI边界情况。

**全量standard模式（最多3000轮、完整搜索）尚未执行，不提供其正式效果保证。**
本文件保持`standard`默认设置，并清空测试输出，避免把快速检查的结果当作正式建模结果。
测试环境为Python 3.12 / Linux / CPU；接口与依赖版本见第1节，Mac实际耗时可能不同。


## 1. 环境与依赖

推荐使用独立环境，Python 3.11/3.12（你的参考稿为Python 3.11）。下面采用已核对的API写法，测试环境为
LightGBM 4.6.0、pandas 2.2.3、NumPy 2.3.5、scikit-learn 1.8.0。
现有环境能运行则无需升级；需复现实测依赖时，在独立环境中取消下一行安装命令的注释，安装后重启kernel。
不要在每次训练前自动升级环境，也不要为此直接覆盖原模型项目的依赖。
Mac如出现libomp动态库缺失，可在终端用 `brew install libomp` 安装OpenMP运行库。


In [1]:
# %pip install "lightgbm==4.6.0" "pandas==2.2.3" "numpy==2.3.5" "scikit-learn==1.8.0" scipy matplotlib joblib packaging

import gc
import hashlib
import inspect
import json
import os
import platform
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
from packaging.version import Version
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, average_precision_score,
                             precision_recall_curve, log_loss, confusion_matrix)
import lightgbm as lgb
from lightgbm import LGBMClassifier


assert Version(sklearn.__version__) >= Version('1.3')
assert Version(lgb.__version__) >= Version('4.0')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.2})
VERSIONS = {
    'python': platform.python_version(), 'numpy': np.__version__,
    'pandas': pd.__version__, 'scipy': scipy.__version__,
    'scikit-learn': sklearn.__version__, 'matplotlib': matplotlib.__version__,
    'joblib': joblib.__version__, 'lightgbm': lgb.__version__
}
display(pd.Series(VERSIONS, name='version'))


python          3.11.15
numpy             2.4.5
pandas            3.0.5
scipy            1.17.1
scikit-learn      1.8.0
matplotlib       3.10.9
joblib            1.5.3
lightgbm          4.7.0
Name: version, dtype: str

## 2. 项目配置

`AR_TOLERANCE=0.005` 是AR的绝对差0.005：入围后优先选择Train–Selection差距较小者。
它是一条预先约定的选择规则，并不保证生产环境泛化。训练搜索串行运行，每个模型内部使用多线程。
不做SMOTE、不默认把负正样本比例设为权重；本项目先比较不加权模型的区分能力。
完整模式最多约17组（LightGBM）/18组（XGBoost），每组最多3000轮；quick约7组、每组最多500轮。
实际耗时取决于CPU和早停，不建议同时运行两份完整notebook。

`EXCLUDE_FEATURES` 是业务排除清单，必须在看Test前确定。年龄、家庭、地域等字段仍可能有合规或代理偏差，
默认排除性别不代表已经完成公平性审查。若业务政策有额外限制，应先修改此处再完整重跑。


In [2]:
MODEL_KIND = 'lightgbm'
MODEL_LOCKED = False
CSV_PATH = '/Users/ganlu/Desktop/home-credit-risk-english/data/application_train.csv' # 例如 Path('/Users/ganlu/Desktop/home-credit-risk-english/data/application_train.csv')
RUN_MODE = 'standard'  # standard / quick / smoke
random_state = 42
N_JOBS = min(8, os.cpu_count() or 1)  # 24GB Mac可按需要改成4；每次只运行一份notebook
USE_FEATURE_ENGINEERING = True
EXCLUDE_FEATURES = ['CODE_GENDER']
AR_TOLERANCE = 0.005
REVIEW_RATE = 0.10  # 演示：固定Selection最高风险约10%的分数阈值，不代表拒贷政策
MAX_DENSE_GB = 2.0  # XGBoost单个dense矩阵的保护上限，不是进程总内存上限

PROFILES = {
    'standard': {'iterations': 3000, 'early_stopping': 150, 'sample_n': None, 'shap_n': 1000, 'bootstrap_n': 200},
    'quick': {'iterations': 500, 'early_stopping': 50, 'sample_n': None, 'shap_n': 300, 'bootstrap_n': 50},
    'smoke': {'iterations': 80, 'early_stopping': 15, 'sample_n': 12000, 'shap_n': 50, 'bootstrap_n': 20}
}
if RUN_MODE not in PROFILES:
    raise ValueError('RUN_MODE只能是standard、quick或smoke。')
PROFILE = PROFILES[RUN_MODE]
SEARCH_ITERATIONS = PROFILE['iterations']
EARLY_STOPPING = PROFILE['early_stopping']
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR = Path.cwd() / 'model_outputs' / MODEL_KIND / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
print('运行模式:', RUN_MODE, '输出目录:', RUN_DIR)
if RUN_MODE != 'standard':
    warnings.warn('当前是检查模式；参数搜索范围和树数预算已缩小，不用于正式效果结论。')


运行模式: standard 输出目录: /Users/ganlu/Desktop/home-credit-risk-english/machinelearning_final/model_outputs/lightgbm/20260830T091052_163659Z


## 3. 读取与核验数据

沿用原始 `TARGET` 定义，1代表原数据中的违约/偿付困难标记，不能自行改称“未来12个月90+逾期”。
本次附件实测为 **307,511行、122列、24,825个1标签（8.0729%）**，客户ID唯一、无整行重复。
代码会再次检查实际读入文件；标签缺失、非法标签或重复客户ID直接停止，不静默删行。


In [3]:
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


if CSV_PATH is None:
    candidates = [
        Path(folder) / name
        for folder in ['.', 'upload', '../upload', 'data', '../data']
        for name in ['application_train.csv', 'application_train(4).csv']
    ]
    found = list(dict.fromkeys(p.resolve() for p in candidates if p.is_file()))
    if len(found) != 1:
        raise FileNotFoundError(f'找到{len(found)}个候选CSV，请在配置单元明确填写CSV_PATH：{found}')
    CSV_PATH = found[0]
CSV_PATH = Path(CSV_PATH).expanduser().resolve()
df = pd.read_csv(CSV_PATH, low_memory=False)
DATA_SHA256 = file_sha256(CSV_PATH)
required = {'TARGET', 'SK_ID_CURR'}
if not required.issubset(df.columns) or not df.columns.is_unique:
    raise ValueError('缺少TARGET/SK_ID_CURR或列名不唯一。')
if df['TARGET'].isna().any() or set(df['TARGET'].unique()) != {0, 1}:
    raise ValueError('TARGET必须非空且同时包含0和1。')
if df['SK_ID_CURR'].isna().any() or not df['SK_ID_CURR'].is_unique:
    raise ValueError('本流程要求一客户一行；重复客户必须先确定分组/时间切分规则，不能随机拆散。')
original_n = len(df)
source_rows = np.arange(original_n)
overview = pd.Series({
    'rows': original_n, 'columns': df.shape[1], 'bad_n': int(df['TARGET'].sum()),
    'bad_rate': df['TARGET'].mean(), 'duplicate_rows': int(df.duplicated().sum()),
    'unique_id': df['SK_ID_CURR'].nunique(),
    'employment_special_n': int(df['DAYS_EMPLOYED'].eq(365243).sum())
})
display(overview)

# 仅smoke抽样；standard/quick保留全部数据、全部行顺序。
if PROFILE['sample_n'] is not None and len(df) > PROFILE['sample_n']:
    keep, _ = train_test_split(source_rows, train_size=PROFILE['sample_n'],
                               stratify=df['TARGET'], random_state=random_state)
    source_rows = np.sort(keep)
    df = df.iloc[source_rows].reset_index(drop=True)
    print('SMOKE程序测试样本数:', len(df), '（非正式建模样本）')


rows                   307511.000000
columns                   122.000000
bad_n                   24825.000000
bad_rate                    0.080729
duplicate_rows              0.000000
unique_id              307511.000000
employment_special_n    55374.000000
dtype: float64

## 4. 先切分，再拟合所有数据驱动规则

Train负责训练和预处理规则；Selection负责早停和参数选择，因此不能再称独立测试集。
Test此时只保留原始行，不生成特征报告、不评估分数。与参考稿一样采用两次`train_test_split`。
只有在CSV内容、行顺序、随机种子和抽样模式相同的情况下，划分才完全一致。


In [4]:
y = df['TARGET'].astype('int8')
raw_columns = [c for c in df.columns if c not in ['TARGET', 'SK_ID_CURR'] + EXCLUDE_FEATURES]
raw = df.loc[:, raw_columns]
idx = np.arange(len(df))
train_idx, temp_idx = train_test_split(idx, test_size=0.30, stratify=y, random_state=random_state)
selection_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=y.iloc[temp_idx], random_state=random_state)
assert len(set(train_idx) & set(selection_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(selection_idx) & set(test_idx)) == 0
assert len(train_idx) + len(selection_idx) + len(test_idx) == len(df)

raw_train = raw.iloc[train_idx].copy()
raw_selection = raw.iloc[selection_idx].copy()
raw_test = raw.iloc[test_idx].copy()  # 封存到锁模后
y_train = y.iloc[train_idx].copy()
y_selection = y.iloc[selection_idx].copy()
y_test = y.iloc[test_idx].copy()

split_label = np.empty(len(df), dtype=object)
split_label[train_idx] = 'Train'
split_label[selection_idx] = 'Selection'
split_label[test_idx] = 'Test'
split_manifest = pd.DataFrame({'source_row': source_rows, 'SK_ID_CURR': df['SK_ID_CURR'], 'sample': split_label})
split_bytes = split_manifest.to_csv(index=False).encode('utf-8')
SPLIT_SHA256 = hashlib.sha256(split_bytes).hexdigest()
split_manifest.to_csv(RUN_DIR / 'split_manifest.csv', index=False)
display(pd.DataFrame({
    'sample': ['Train', 'Selection', 'Test (sealed)'],
    'n': [len(train_idx), len(selection_idx), len(test_idx)],
    'bad_rate': [y_train.mean(), y_selection.mean(), np.nan]
}))
print('split_sha256:', SPLIT_SHA256)


,sample,n,bad_rate
0,Train,215257,0.080727
1,Selection,46127,0.080734
2,Test (sealed),46127,NaN


split_sha256: 760d998abf20a084447d7bd8107166c28f7199546d9769fd9902236189f5cf5f


## 5. 固定规则的特征工程

保留LR原稿的年限与比例变量，并增加`EMPLOYED_SPECIAL_FLAG`保留特殊值的信息。
年限由负天数转换；`CREDIT_ANNUITY_RATIO`只是金额比例，不直接等同贷款期限。
`ANNUITY_INCOME_RATIO`也不是已核验口径的DSR；金额周期必须先核对。
高收入等极端值不凭直觉裁剪，缺失留给树模型。以后若要加分位截尾，边界只能从Train学习并保存。


In [ ]:
def safe_div(a, b):
    # 分母为0或缺失，结果保留NaN，不能把“无法计算”当成数值0。
    return a.div(b.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)


def prepare_features(raw, use_feature_engineering=True):
    """只做逐行、固定规则的加工，不使用标签或全表统计量。"""
    data = raw.copy()
    numeric = data.select_dtypes(include='number').columns
    data[numeric] = data[numeric].replace([np.inf, -np.inf], np.nan)
    # 整理数据块再添加新列，避免宽表逐列赋值后的碎片化警告。
    data = data.copy()

    # 延续LR原稿：365243是特殊占位值，不是1000年的真实工作年限。
    if 'DAYS_EMPLOYED' in data:
        special = data['DAYS_EMPLOYED'].eq(365243)
        data['EMPLOYED_SPECIAL_FLAG'] = special.astype('int8')
        data['DAYS_EMPLOYED'] = data['DAYS_EMPLOYED'].mask(special)

    if use_feature_engineering:
        day_map = {
            'DAYS_BIRTH': 'AGE_YEARS',
            'DAYS_EMPLOYED': 'EMPLOYED_YEARS',
            'DAYS_REGISTRATION': 'REGISTRATION_YEARS',
            'DAYS_ID_PUBLISH': 'ID_PUBLISH_YEARS',
            'DAYS_LAST_PHONE_CHANGE': 'PHONE_CHANGE_YEARS'
        }
        for old, new in day_map.items():
            data[new] = -pd.to_numeric(data[old], errors='raise') / 365.25

        ratios = {
            'CREDIT_INCOME_RATIO': ('AMT_CREDIT', 'AMT_INCOME_TOTAL'),
            'ANNUITY_INCOME_RATIO': ('AMT_ANNUITY', 'AMT_INCOME_TOTAL'),
            'CREDIT_ANNUITY_RATIO': ('AMT_CREDIT', 'AMT_ANNUITY'),
            'GOODS_CREDIT_RATIO': ('AMT_GOODS_PRICE', 'AMT_CREDIT'),
            'INCOME_PER_PERSON': ('AMT_INCOME_TOTAL', 'CNT_FAM_MEMBERS'),
            'EMPLOYED_AGE_RATIO': ('EMPLOYED_YEARS', 'AGE_YEARS')
        }
        for name, (a, b) in ratios.items():
            data[name] = safe_div(data[a], data[b])
        data = data.drop(columns=list(day_map))

    # 不做全样本中位数填补、标准化、WOE或任意截尾。
    numeric = data.select_dtypes(include='number').columns
    data[numeric] = data[numeric].astype('float32').replace([np.inf, -np.inf], np.nan)
    return data.copy()


def category_text(s):
    # 给真实类别加前缀，避免真实值恰好叫“__MISSING__”时与缺失混淆。
    return ('V:' + s.astype('string')).fillna('__MISSING__').astype(object)


def validate_raw_schema(raw, spec):
    """允许额外列和列顺序变化；缺少约定输入列或非法数值则明确报错。"""
    if not isinstance(raw, pd.DataFrame) or not raw.columns.is_unique:
        raise ValueError('输入必须是列名唯一的DataFrame。')
    missing = sorted(set(spec['raw_columns']) - set(raw.columns))
    if missing:
        raise ValueError(f'缺少模型输入列：{missing}')
    data = raw.loc[:, spec['raw_columns']].copy()
    for col in spec['raw_numeric']:
        converted = pd.to_numeric(data[col], errors='coerce')
        invalid = data[col].notna() & converted.isna()
        if invalid.any():
            raise ValueError(f'{col} 有 {int(invalid.sum())} 个无法转换为数值的非空值。')
        data[col] = converted
    return data


In [ ]:
F_train = prepare_features(raw_train, USE_FEATURE_ENGINEERING)
F_selection = prepare_features(raw_selection, USE_FEATURE_ENGINEERING)
assert F_train.index.equals(y_train.index)
assert F_selection.index.equals(y_selection.index)
display(F_train[['EMPLOYED_SPECIAL_FLAG'] + (['AGE_YEARS', 'EMPLOYED_YEARS', 'CREDIT_INCOME_RATIO'] if USE_FEATURE_ENGINEERING else [])].head())


## 6. Train上的基础质量筛查与剔除记录

沿用CatBoost稿的99.5%近常量规则，但仅一个非缺失取值且存在缺失时，不直接按常量剔除，
因为“是否缺失”可能有信息。近常量规则仍可能删除稀有但有业务意义的事件，投产前应复核剔除清单。
高缺失率只提示，不自动删除。缺失与非缺失坏率仅作Train诊断，不是因果关系。
树模型不需要为了满足LR假设而执行WOE单调分箱、IV门槛、VIF或P-value筛选。


In [ ]:
def scan_feature_quality(X, y):
    if not X.index.equals(y.index):
        raise ValueError('特征与标签索引不一致。')
    rows = []
    for col in X.columns:
        s = X[col]
        miss = s.isna()
        is_cat = not pd.api.types.is_numeric_dtype(s)
        top_frequency = float(s.value_counts(dropna=False, normalize=True).iloc[0])
        n_unique = int(s.nunique(dropna=True))
        reason = ''
        if miss.all():
            reason = 'all missing'
        elif s.nunique(dropna=False) <= 1:
            reason = 'constant'
        elif top_frequency > 0.995:
            reason = 'quasi constant > 99.5%'
        elif is_cat and n_unique > 1000 and n_unique / len(s) > 0.95:
            reason = 'ID-like categorical'
        rows.append({
            'feature': col, 'dtype': str(s.dtype), 'is_categorical': is_cat,
            'nunique': n_unique, 'missing_rate': float(miss.mean()),
            'top_frequency': top_frequency,
            'bad_rate_missing': y.loc[miss].mean() if miss.any() else np.nan,
            'bad_rate_nonmissing': y.loc[~miss].mean() if (~miss).any() else np.nan,
            'drop': bool(reason), 'reason': reason,
            'warning': 'high missing > 95%' if miss.mean() > 0.95 else ''
        })
    return pd.DataFrame(rows)


feature_report = scan_feature_quality(F_train, y_train)
basic_drop = feature_report.loc[feature_report['drop'], ['feature', 'reason']]
final_features = feature_report.loc[~feature_report['drop'], 'feature'].tolist()
assert final_features, '基础筛查后没有可用特征。'
cat_cols = [f for f in final_features if not pd.api.types.is_numeric_dtype(F_train[f])]
num_cols = [f for f in final_features if f not in cat_cols]
print('加工后特征数:', F_train.shape[1])
print('基础筛查剔除:', len(basic_drop), '保留:', len(final_features), '类别变量:', len(cat_cols))
display(basic_drop)
display(feature_report.sort_values('missing_rate', ascending=False).head(20))


## 7. LightGBM原生类别变量：只在Train学习类别表

类别编码只是标识，不把“类别编号大小”当成连续关系。Selection/Test/新客户都用同一套Train类别表；未见类别进入原生缺失路径。数值NaN保留，真实0保留。
不需要One-Hot，也不需要标准化。


In [ ]:
category_levels = {
    c: sorted(set(category_text(F_train[c]).unique().tolist()) | {'__MISSING__'})
    for c in cat_cols
}
preprocess_spec = {
    'schema_version': 1, 'model_kind': MODEL_KIND,
    'use_feature_engineering': USE_FEATURE_ENGINEERING,
    'raw_columns': raw_columns,
    'raw_numeric': raw_train.select_dtypes(include='number').columns.tolist(),
    'features': final_features, 'numeric': num_cols, 'categorical': cat_cols,
    'category_levels': category_levels, 'excluded_features': EXCLUDE_FEATURES,
    'max_dense_gb': MAX_DENSE_GB
}


In [ ]:
def transform_model_frame(features, spec, encoder=None):
    out = features.loc[:, spec['features']].copy()
    for col in spec['numeric']:
        out[col] = pd.to_numeric(out[col], errors='raise').astype('float32')
    for col in spec['categorical']:
        # 全部集合沿用Train的类别表；未见类别自动成为NaN，交给原生缺失分支。
        out[col] = pd.Categorical(category_text(out[col]), categories=spec['category_levels'][col])
    return out


def transform_raw(raw, spec, encoder=None):
    checked = validate_raw_schema(raw, spec)
    features = prepare_features(checked, spec['use_feature_engineering'])
    return transform_model_frame(features, spec, encoder)


In [ ]:
encoder = None
X_train = transform_model_frame(F_train, preprocess_spec)
X_selection = transform_model_frame(F_selection, preprocess_spec)
feature_mapping = pd.DataFrame({'model_feature': final_features, 'original_feature': final_features})
print('训练矩阵:', X_train.shape, '内存MiB:', round(X_train.memory_usage(deep=True).sum() / 1024**2, 1))
assert all(list(X_train[c].cat.categories) == list(X_selection[c].cat.categories) for c in cat_cols)


## 8. Train–Selection特征PSI：只诊断，不据此调Test

连续变量按Train分位点建箱，边界扩展到正负无穷；离散数值按取值；类别按Train词表。
缺失单独成箱，未见类别单独统计。全部分箱规则保存供后续监控复用。
随机切分中的低PSI不能证明跨时间稳定；0.10/0.25只作展示用提醒，不是通用监管标准。


In [ ]:
def make_psi_spec(s, bins=10):
    """边界/类别只从参考集确定；低基数数值按取值分组，防止0/1被合成一箱。"""
    if pd.api.types.is_numeric_dtype(s) and s.nunique(dropna=True) > bins:
        valid = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
        cuts = np.unique(valid.quantile(np.linspace(0, 1, bins + 1)[1:-1]).to_numpy())
        spec = {'kind': 'numeric', 'cuts': [float(x) for x in cuts]}
        labels = [f'B{i}' for i in range(len(cuts) + 1)] + ['__MISSING__']
    else:
        # 数值离散列用float64统一字符串格式，避免float32与Python float的表示差异。
        ss = s.astype('float64') if pd.api.types.is_numeric_dtype(s) else s
        levels = sorted(category_text(ss).unique().tolist())
        spec = {'kind': 'discrete', 'levels': levels,
                'numeric': bool(pd.api.types.is_numeric_dtype(s))}
        labels = sorted(set(levels + ['__MISSING__', '__UNSEEN__']))
    spec['labels'] = labels
    counts = psi_labels(s, spec).value_counts().reindex(labels, fill_value=0)
    spec['reference_counts'] = [int(x) for x in counts]
    return spec


def psi_labels(s, spec):
    if spec['kind'] == 'numeric':
        x = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan)
        bins = [-np.inf] + spec['cuts'] + [np.inf]
        labels = [f'B{i}' for i in range(len(bins) - 1)]
        return pd.cut(x, bins=bins, labels=labels, include_lowest=True).astype(object).fillna('__MISSING__')
    ss = pd.to_numeric(s, errors='raise').astype('float64') if spec['numeric'] else s
    text = category_text(ss)
    return text.where(text.isin(spec['levels']) | text.eq('__MISSING__'), '__UNSEEN__')


def calculate_psi(other_s, spec, alpha=0.5):
    # 计数加伪计数后重新归一化，两个分布的占比之和都为1。
    ref = np.asarray(spec['reference_counts'], dtype=float)
    cur = psi_labels(other_s, spec).value_counts().reindex(spec['labels'], fill_value=0).to_numpy(float)
    if ref.sum() == 0 or cur.sum() == 0:
        raise ValueError('PSI不能使用空样本。')
    p = (ref + alpha) / (ref.sum() + alpha * len(ref))
    q = (cur + alpha) / (cur.sum() + alpha * len(cur))
    return float(np.sum((q - p) * np.log(q / p)))


psi_specs = {f: make_psi_spec(F_train[f]) for f in final_features}
psi_selection = pd.DataFrame([
    {'feature': f, 'psi_selection': calculate_psi(F_selection[f], psi_specs[f])}
    for f in final_features
]).sort_values('psi_selection', ascending=False)
display(psi_selection.head(20))


## 9. 统一指标

**AUC越高越好，AR=2×AUC−1，KS=max(TPR−FPR)**，全部基于“分数越高风险越高”的方向。
不把负AR取绝对值掩盖反向预测。AP是Average Precision，用于观察少数类识别；它不是梯形积分PR-AUC。
Logloss仅补充诊断，没有在这里拟合校准器。Accuracy与固定0.5阈值不作为主选模标准。


In [ ]:
def get_metrics(y, pred):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(pred, dtype=float)
    if len(y) != len(pred) or set(np.unique(y)) != {0, 1}:
        raise ValueError('评估集必须同时包含0/1，且预测与标签长度相等。')
    if not np.isfinite(pred).all() or ((pred < 0) | (pred > 1)).any():
        raise ValueError('预测必须是有限的0~1数值。')
    auc = roc_auc_score(y, pred)
    fpr, tpr, _ = roc_curve(y, pred)
    return {
        'n': len(y), 'bad_rate': y.mean(), 'AUC': auc,
        'AR': 2 * auc - 1, 'KS': np.max(tpr - fpr),
        'AP': average_precision_score(y, pred),
        'Logloss': log_loss(y, pred, labels=[0, 1])
    }


def evaluate_predictions(predictions, labels):
    return pd.DataFrame([
        {'sample': name, **get_metrics(labels[name], pred)}
        for name, pred in predictions.items()
    ]).set_index('sample')


## 10. 算法参数与早停接口

LightGBM使用叶子优先生长，重点同时约束num_leaves、max_depth和min_child_samples。设置subsample时必须同时启用subsample_freq；否则采样比例可能不生效。

| 参数 | 这里的作用 |
|---|---|
| num_leaves / max_depth | 树容量上限 |
| min_child_samples | 叶节点样本规模约束 |
| learning_rate + n_estimators | 每棵树的步长与最大轮数 |
| reg_lambda / reg_alpha | L2 / L1正则 |
| min_split_gain | 分裂至少需要的收益 |
| subsample / colsample_bytree | 样本与列采样 |
| cat_smooth | 类别统计平滑 |

优化目标为二分类Logloss，早停只监控Selection AUC。
LightGBM 4.x通过callbacks传入early_stopping，best_iteration_已经是应使用的轮数，不再加1。


In [ ]:
BASE_PARAMS = {
    'num_leaves': 31, 'max_depth': 6, 'min_child_samples': 200,
    'learning_rate': 0.03, 'reg_lambda': 5.0, 'reg_alpha': 0.0,
    'subsample': 0.80, 'colsample_bytree': 0.90,
    'min_split_gain': 0.0, 'cat_smooth': 10.0
}


def make_model(params, n_estimators, early_stop=False):
    return LGBMClassifier(
        objective='binary', metric='auc', boosting_type='gbdt',
        n_estimators=int(n_estimators), random_state=random_state,
        n_jobs=N_JOBS, verbosity=-1, importance_type='gain',
        subsample_freq=1,  # 不为0，subsample才真正生效
        max_bin=255, deterministic=True, force_col_wise=True,
        use_missing=True, zero_as_missing=False,
        scale_pos_weight=1.0, **params
    )


def fit_with_early_stopping(model):
    model.fit(
        X_train, y_train, categorical_feature=cat_cols,
        eval_set=[(X_train, y_train), (X_selection, y_selection)],
        eval_names=['Train', 'Selection'], eval_metric='auc',
        callbacks=[lgb.early_stopping(EARLY_STOPPING, first_metric_only=True, verbose=False),
                   lgb.log_evaluation(period=0)]
    )


def get_best_tree_count(model):
    return int(model.best_iteration_ or model.n_estimators_)


def get_trained_tree_count(model):
    # Booster可能已裁剪到最佳轮次，学习曲线长度才是实际尝试的轮数。
    return len(model.evals_result_['Selection']['auc'])


def predict_model(model, X, n_trees=None):
    return model.predict_proba(X, num_iteration=n_trees)[:, 1]


## 11. 共用搜索函数与choose_best

每组模型只用Train训练，用Selection早停；记录AR、KS、gap、实际训练轮数和耗时。
`best_iteration`在两份notebook中统一表示**实际使用的树轮数**。
同一参数组合复用本次会话缓存；每个运行目录独立，避免把别的样本或配置的搜索记录误用于本次。
参数改变后应重启kernel并从头运行，不要沿用旧缓存。


In [ ]:
# 只缓存参数和指标，不同时保留几十个模型，减少内存占用。
search_cache = {}
search_rows = []


def run_candidate(params, stage='search', return_model=False):
    key = json.dumps(params, sort_keys=True)
    if key in search_cache and not return_model:
        return search_cache[key].copy()

    start = time.perf_counter()
    model = make_model(params, SEARCH_ITERATIONS, early_stop=True)
    fit_with_early_stopping(model)
    best_n = get_best_tree_count(model)
    train = get_metrics(y_train, predict_model(model, X_train, best_n))
    selection = get_metrics(y_selection, predict_model(model, X_selection, best_n))
    trained_n = get_trained_tree_count(model)
    row = {
        **params, 'stage': stage, 'best_iteration': best_n,
        'trained_iterations': trained_n,
        'reached_cap': trained_n >= SEARCH_ITERATIONS,
        'train_auc': train['AUC'], 'selection_auc': selection['AUC'],
        'train_ar': train['AR'], 'selection_ar': selection['AR'],
        'ar_gap': abs(train['AR'] - selection['AR']),
        'signed_ar_gap': train['AR'] - selection['AR'],
        'train_ks': train['KS'], 'selection_ks': selection['KS'],
        'ks_gap': abs(train['KS'] - selection['KS']),
        'selection_ap': selection['AP'],
        'seconds': time.perf_counter() - start
    }
    search_cache[key] = row.copy()
    search_rows.append(row.copy())
    # 每完成一组就写检查点；中断不会抹掉已完成组的结果。
    pd.DataFrame(search_rows).to_csv(RUN_DIR / 'search_checkpoint.csv', index=False, encoding='utf-8-sig')
    print(f"{stage}: AR={row['selection_ar']:.4f}, gap={row['ar_gap']:.4f}, "
          f"best={best_n}, {row['seconds']:.1f}s", flush=True)
    if return_model:
        return model, row
    del model
    gc.collect()
    return row


def run_search(param_list, stage):
    rows = []
    for i, params in enumerate(param_list, 1):
        print(f'{stage} {i}/{len(param_list)}: {params}', flush=True)
        rows.append(run_candidate(params, stage=stage))
    return pd.DataFrame(rows).sort_values(['selection_ar', 'ar_gap'], ascending=[False, True]).reset_index(drop=True)


def choose_best(result, ar_tolerance=AR_TOLERANCE):
    if result.empty or not np.isfinite(result['selection_ar']).all():
        raise ValueError('没有可用的候选结果。')
    best_ar = result['selection_ar'].max()
    shortlist = result.loc[result['selection_ar'] >= best_ar - ar_tolerance].copy()
    # 先入围AR接近最优的候选，再看gap，最后偏好容量较小/树数较少的模型。
    shortlist['capacity'] = shortlist['num_leaves'] if MODEL_KIND == 'lightgbm' else 2 ** shortlist['max_depth']
    return shortlist.sort_values(
        ['ar_gap', 'capacity', 'best_iteration', 'selection_ar'],
        ascending=[True, True, True, False]
    ).iloc[0]


def row_to_params(row):
    # pandas一行Series可能把整数参数变成float，显式恢复原参数类型。
    return {key: type(value)(row[key]) for key, value in BASE_PARAMS.items()}


## 12. Baseline：先看固定参数的基准表现

Baseline也是最终候选之一，不能假定调参后必然优于它。
学习曲线显示Train与Selection AUC随迭代变化；不展示Test学习曲线。


In [ ]:
baseline_model, baseline_row = run_candidate(BASE_PARAMS, stage='Baseline', return_model=True)
display(pd.DataFrame([baseline_row]))
history = baseline_model.evals_result_
fig, ax = plt.subplots(figsize=(8, 4))
for sample, values in history.items():
    auc_values = values['auc']
    ax.plot(np.arange(1, len(auc_values) + 1), auc_values, label=sample)
ax.axvline(baseline_row['best_iteration'], color='black', linestyle='--', label='Best iteration')
ax.set(xlabel='Boosting iteration', ylabel='AUC', title='LightGBM baseline learning curve')
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / 'baseline_learning_curve.png', bbox_inches='tight')
plt.show()
del baseline_model
_ = gc.collect()


## 13. 第一轮：叶子数、树深度和叶节点样本数

先固定学习率、正则与采样，定位合适的复杂度范围。
这里是有预算的候选配置搜索，不是所有组合的穷举，也不保证全局最优。


In [ ]:
complexity_profiles = [(15, 5, 100), (31, 6, 100), (31, 6, 300), (63, 7, 300)]
if RUN_MODE != 'standard':
    complexity_profiles = [(15, 5, 100), (31, 6, 300)]
complexity_candidates = [
    {**BASE_PARAMS, 'num_leaves': leaves, 'max_depth': depth, 'min_child_samples': minimum}
    for leaves, depth, minimum in complexity_profiles
]
complexity_result = run_search(complexity_candidates, 'Round1')
complexity_best = choose_best(complexity_result)
display(complexity_result)
display(complexity_best)


## 14. 第二轮：保留复杂度候选，联合学习率与L2

完整模式保留第一轮选中的配置与另一组Selection AR最高的配置，避免只锁死一个复杂度。
每种复杂度分别测试学习率/L2的成对设置。快速模式只保留选中的配置以减少运行时间。
学习率更低通常需要更多轮数；达到上限会记录`reached_cap`，不能误以为已完成早停。


In [ ]:
anchors = [row_to_params(complexity_best)]
if RUN_MODE == 'standard':
    for _, row in complexity_result.iterrows():
        params = row_to_params(row)
        if params != anchors[0]:
            anchors.append(params)
            break
lr_l2_pairs = [(0.02, 5.0), (0.03, 3.0), (0.03, 10.0), (0.05, 5.0)]
if RUN_MODE != 'standard':
    lr_l2_pairs = [(0.05, 5.0), (0.07, 10.0)]
fine_candidates = [
    {**anchor, 'learning_rate': lr, 'reg_lambda': l2}
    for anchor in anchors for lr, l2 in lr_l2_pairs
]
fine_result = run_search(fine_candidates, 'Round2')
fine_best = choose_best(pd.concat([complexity_result, fine_result], ignore_index=True))
display(fine_result)
display(fine_best)


## 15. 第三轮：采样、L1与分裂门槛

以第二轮后的入选配置为中心，检验更强的随机化或正则。
最后从**Baseline＋全部三轮**统一选择，不直接把最后一轮的赢家当最终模型。
Selection已经被反复用于选择，它的数值通常偏乐观；Test只保留给最终审计。


In [ ]:
anchor = row_to_params(fine_best)
regularization_grid = [
    (0.70, 0.80, 0.0, 0.0),
    (0.80, 0.90, 1.0, 0.05),
    (0.90, 1.00, 1.0, 0.0),
    (0.80, 0.80, 2.0, 0.10)
]
if RUN_MODE != 'standard':
    regularization_grid = regularization_grid[:2]
regularization_candidates = [
    {**anchor, 'subsample': sub, 'colsample_bytree': col, 'reg_alpha': alpha,
     'min_split_gain': split_gain, 'cat_smooth': 20.0}
    for sub, col, alpha, split_gain in regularization_grid
]
regularization_result = run_search(regularization_candidates, 'Round3')
all_search_result = pd.DataFrame(search_rows).drop_duplicates(subset=list(BASE_PARAMS)).reset_index(drop=True)
best_row = choose_best(all_search_result)
display(all_search_result.sort_values('selection_ar', ascending=False))
display(best_row)
if all_search_result['reached_cap'].any():
    warnings.warn('部分候选达到最大轮数，可能还未收敛；正式运行应在打开Test前确认迭代预算充足。')


## 16. 锁定参数和树数，仅在Train上重训

不把Selection并回Train，以保持与参考CatBoost稿及原始三分样本的评估口径一致。
固定树数后不再early stopping，不查看Test来改参数。若未来决定Train＋Selection重训，
那是另一个模型版本，不能把本次Test结果直接作为新模型的验证结果。


In [ ]:
best_params = row_to_params(best_row)
best_iterations = int(best_row['best_iteration'])
final_model = make_model(best_params, best_iterations, early_stop=False)
final_model.fit(X_train, y_train, categorical_feature=cat_cols)
pred_train = predict_model(final_model, X_train, best_iterations)
pred_selection = predict_model(final_model, X_selection, best_iterations)
refit_selection = get_metrics(y_selection, pred_selection)
if abs(refit_selection['AR'] - float(best_row['selection_ar'])) > 1e-5:
    raise RuntimeError('固定轮数重训未复现候选表现，请在打开Test前检查随机种子、参数和环境。')

# 策略演示阈值仅从Selection确定；没有在Test上寻找最优KS点或调审核比例。
review_threshold = float(np.quantile(pred_selection, 1 - REVIEW_RATE))
locked_config = {
    'model_kind': MODEL_KIND, 'run_mode': RUN_MODE, 'random_state': random_state,
    'parameters': best_params, 'iterations': best_iterations,
    'training_sample': 'Train only', 'calibrated': False,
    'review_rate_target_demo': REVIEW_RATE, 'review_threshold_demo': review_threshold,
    'data_sha256': DATA_SHA256, 'split_sha256': SPLIT_SHA256,
    'feature_count': len(final_features), 'encoded_feature_count': X_train.shape[1],
    'versions': VERSIONS, 'locked_utc': datetime.now(timezone.utc).isoformat()
}
(RUN_DIR / 'locked_config.json').write_text(json.dumps(locked_config, ensure_ascii=False, indent=2), encoding='utf-8')
MODEL_LOCKED = True
print('模型已锁定。树轮数:', best_iterations, 'Selection AR:', refit_selection['AR'])
display(pd.Series(best_params))


## 17. 打开Test：最终AUC / AR / KS表

**从这里开始禁止依据Test表现删特征、改参数、改早停轮数。**如发现需改模型，应启动新开发轮次并保留新的独立验证数据。
三个集合的预测各计算一次，后面的图表全部复用这些分数。


In [ ]:
assert MODEL_LOCKED
F_test = prepare_features(validate_raw_schema(raw_test, preprocess_spec), USE_FEATURE_ENGINEERING)
X_test = transform_model_frame(F_test, preprocess_spec, encoder)
pred_test = predict_model(final_model, X_test, best_iterations)
predictions = {'Train': pred_train, 'Selection': pred_selection, 'Test': pred_test}
labels = {'Train': y_train, 'Selection': y_selection, 'Test': y_test}
final_result = evaluate_predictions(predictions, labels)
display(final_result)
gap_table = pd.DataFrame({
    'comparison': ['Train-Selection', 'Train-Test'],
    'AR_gap_signed': [final_result.loc['Train', 'AR'] - final_result.loc[s, 'AR'] for s in ['Selection', 'Test']],
    'KS_gap_signed': [final_result.loc['Train', 'KS'] - final_result.loc[s, 'KS'] for s in ['Selection', 'Test']]
})
display(gap_table)


## 18. ROC、KS与PR诊断

KS曲线的阈值仅用于展示，不把Test上达到最大KS的阈值拿回去设置策略。
不使用“Train与Test差距小”单独判定模型优良，还要看绝对区分能力、样本量和业务可解释性。


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for sample, pred in predictions.items():
    fpr, tpr, _ = roc_curve(labels[sample], pred)
    axes[0].plot(fpr, tpr, label=f"{sample} (AUC={final_result.loc[sample, 'AUC']:.3f})")
axes[0].plot([0, 1], [0, 1], '--', color='gray')
axes[0].set(title='ROC', xlabel='False positive rate', ylabel='True positive rate')
axes[0].legend(fontsize=8)
fpr, tpr, thresholds = roc_curve(y_test, pred_test)
finite = np.isfinite(thresholds)
axes[1].plot(thresholds[finite], (tpr - fpr)[finite], color='#167D8D')
axes[1].set(title='Test KS by threshold (diagnostic)', xlabel='Raw probability threshold', ylabel='TPR - FPR')
precision, recall, _ = precision_recall_curve(y_test, pred_test)
axes[2].plot(recall, precision, color='#DA8244', label=f"AP={final_result.loc['Test', 'AP']:.3f}")
axes[2].axhline(y_test.mean(), linestyle='--', color='gray', label='Test bad rate')
axes[2].set(title='Test precision-recall', xlabel='Recall', ylabel='Precision')
axes[2].legend(fontsize=8)
fig.suptitle('LightGBM — locked model', y=1.02)
fig.tight_layout()
fig.savefig(RUN_DIR / 'final_diagnostics.png', bbox_inches='tight')
plt.show()


## 19. 固定模型的Test不确定性

对Test中的好/坏样本分别有放回抽样，形成分层bootstrap的95%分位区间。
这是**给定当前已训练模型和当前抽样方式**的样本不确定性，不包含重新训练、选模或跨时期变化的不确定性。


In [ ]:
def bootstrap_auc_ar(y, pred, n_boot=200, seed=42):
    yy, pp = np.asarray(y), np.asarray(pred)
    good, bad = np.flatnonzero(yy == 0), np.flatnonzero(yy == 1)
    rng = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_boot):
        ii = np.r_[rng.choice(good, len(good), replace=True), rng.choice(bad, len(bad), replace=True)]
        aucs.append(roc_auc_score(yy[ii], pp[ii]))
    lo, hi = np.quantile(aucs, [0.025, 0.975])
    auc = roc_auc_score(yy, pp)
    return pd.DataFrame({'metric': ['AUC', 'AR'], 'estimate': [auc, 2*auc-1],
                         'lower_95': [lo, 2*lo-1], 'upper_95': [hi, 2*hi-1], 'bootstrap_n': n_boot})


test_confidence_interval = bootstrap_auc_ar(y_test, pred_test, PROFILE['bootstrap_n'], random_state)
display(test_confidence_interval)


## 20. 风险十分组、Lift与固定审核阈值演示

十分组是对Test按风险降序的**事后描述性排名**，第1组最高风险，不是校准后的等级。
相同分数可能跨排名组，不能直接把这些分数边界当上线规则。单独的阈值演示使用锁模时从Selection确定的分位点；
若有并列分数，实际命中率不一定恰好10%。输出只表示进入人工复核的演示标记，不自动作信贷决定。


In [ ]:
def risk_decile_table(y, pred, groups=10):
    ranked = pd.DataFrame({'target': np.asarray(y), 'raw_probability': np.asarray(pred)})
    ranked = ranked.sort_values('raw_probability', ascending=False, kind='stable').reset_index(drop=True)
    ranked['rank'] = np.arange(1, len(ranked) + 1)
    ranked['risk_group'] = np.minimum((np.arange(len(ranked)) * groups // len(ranked)) + 1, groups)
    table = ranked.groupby('risk_group', observed=True).agg(
        rank_min=('rank', 'min'), rank_max=('rank', 'max'), n=('target', 'size'),
        bad_n=('target', 'sum'), bad_rate=('target', 'mean'),
        probability_min=('raw_probability', 'min'), probability_max=('raw_probability', 'max')
    ).reset_index()
    table['lift'] = table['bad_rate'] / ranked['target'].mean()
    table['cumulative_bad_capture'] = table['bad_n'].cumsum() / ranked['target'].sum()
    return table


decile_table = risk_decile_table(y_test, pred_test)
display(decile_table)
test_review = pred_test >= review_threshold
tn, fp, fn, tp = confusion_matrix(y_test, test_review.astype(int), labels=[0, 1]).ravel()
review_table = pd.Series({
    'threshold_from_selection': review_threshold,
    'selection_flag_rate': (pred_selection >= review_threshold).mean(),
    'test_flag_rate': test_review.mean(),
    'test_bad_capture': tp / (tp + fn),
    'test_flagged_bad_rate': tp / (tp + fp) if tp + fp else np.nan,
    'test_good_flag_rate': fp / (fp + tn),
    'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp
}, name='manual_review_demo')
display(review_table)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(decile_table['risk_group'], decile_table['bad_rate'], color='#167D8D')
ax.axhline(y_test.mean(), color='#DA8244', linestyle='--', label='Overall Test bad rate')
ax.set(xlabel='Test risk group (1 = highest risk)', ylabel='Observed bad rate', title='Test risk deciles')
ax.set_xticks(range(1, 11))
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / 'risk_deciles.png', bbox_inches='tight')
plt.show()


## 21. 锁模后的Test稳定性与输入漂移

Test特征PSI仍使用第8节的Train分箱，不重新估边界。分数PSI使用Train预测的分位点。
Train分数可能因拟合而更极端，因此Train–Test分数PSI不等同生产时间漂移。
同时记录缺失率和未见类别比例；这些诊断不能用于这次Test驱动的删变量。


In [ ]:
psi_test = pd.DataFrame([{'feature': f, 'psi_test': calculate_psi(F_test[f], psi_specs[f])} for f in final_features])
psi_table = psi_selection.merge(psi_test, on='feature', validate='one_to_one')
psi_table['max_psi'] = psi_table[['psi_selection', 'psi_test']].max(axis=1)
psi_table['psi_flag_demo'] = np.select([psi_table['max_psi'] >= 0.25, psi_table['max_psi'] >= 0.10],
                                       ['investigate', 'watch'], default='low')
score_psi_spec = make_psi_spec(pd.Series(pred_train))
score_psi = pd.DataFrame([
    {'comparison': 'Train-Selection', 'PSI': calculate_psi(pd.Series(pred_selection), score_psi_spec)},
    {'comparison': 'Train-Test', 'PSI': calculate_psi(pd.Series(pred_test), score_psi_spec)}
])
input_drift = pd.DataFrame([
    {'feature': f,
     'train_missing_rate': F_train[f].isna().mean(),
     'selection_missing_rate': F_selection[f].isna().mean(),
     'test_missing_rate': F_test[f].isna().mean(),
     'test_unseen_rate': (~category_text(F_test[f]).isin(category_levels[f])).mean() if f in cat_cols else np.nan}
    for f in final_features
])
display(psi_table.sort_values('max_psi', ascending=False).head(25))
display(score_psi)
display(input_drift.loc[input_drift['test_unseen_rate'] > 0])


## 22. 特征重要性：总分裂收益

LightGBM使用gain（全部分裂收益之和）并同时显示split次数。
归一化为百分比方便查看，不是逻辑回归系数、评分权重或因果影响。相关变量可能分摊重要性；
此处只解释锁定模型，不再根据Test解释结果重选特征。


In [ ]:
feature_importance = pd.DataFrame({
    'feature': final_model.booster_.feature_name(),
    'total_gain': final_model.booster_.feature_importance(importance_type='gain'),
    'split_count': final_model.booster_.feature_importance(importance_type='split')
})
total_gain = feature_importance['total_gain'].sum()
feature_importance['gain_pct'] = feature_importance['total_gain'] / total_gain if total_gain > 0 else 0.0
feature_importance = feature_importance.sort_values('total_gain', ascending=False).reset_index(drop=True)
display(feature_importance.head(30))


## 23. 原生Tree SHAP：全局与单客户解释

使用模型内置贡献计算，不依赖额外的`shap`包。随机抽取Test样本，**最后一列是基准项，不能当成特征**。
贡献单位是raw margin/log-odds：`基准项 + 特征贡献之和 = 模型raw margin`，再经过sigmoid才是预测概率。
正贡献推高模型风险，负贡献降低模型风险；不是概率百分点，也不是因果或正式拒贷理由。
直接对应原始入模特征。


In [ ]:
rng = np.random.default_rng(random_state)
shap_positions = rng.choice(len(y_test), size=min(PROFILE['shap_n'], len(y_test)), replace=False)
X_shap = X_test.iloc[shap_positions]
contributions = np.asarray(final_model.booster_.predict(X_shap, pred_contrib=True, num_iteration=best_iterations))
raw_margin = final_model.booster_.predict(X_shap, raw_score=True, num_iteration=best_iterations)

assert contributions.shape == (len(shap_positions), X_train.shape[1] + 1)
shap_values = contributions[:, :-1]
shap_bias = contributions[:, -1]
np.testing.assert_allclose(shap_values.sum(axis=1) + shap_bias, raw_margin, atol=2e-5, rtol=1e-5)
np.testing.assert_allclose(1 / (1 + np.exp(-raw_margin)), pred_test[shap_positions], atol=2e-6, rtol=1e-5)

# 同一原始变量的多列贡献先求和，之后才计算绝对值均值。
raw_shap = np.zeros((len(shap_positions), len(final_features)), dtype=float)
feature_position = {f: i for i, f in enumerate(final_features)}
for j, feature in enumerate(feature_mapping['original_feature']):
    raw_shap[:, feature_position[feature]] += shap_values[:, j]
np.testing.assert_allclose(raw_shap.sum(axis=1), shap_values.sum(axis=1), atol=2e-5, rtol=1e-5)
shap_importance = pd.DataFrame({'feature': final_features, 'mean_abs_shap': np.abs(raw_shap).mean(axis=0)}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
display(shap_importance.head(30))

customer_pos = 0  # 第一个抽样客户，仅作解释示例
individual_explanation = pd.DataFrame({
    'feature': final_features,
    'feature_value': F_test.iloc[shap_positions[customer_pos]][final_features].to_numpy(),
    'shap_log_odds': raw_shap[customer_pos]
})
individual_explanation['abs_shap'] = individual_explanation['shap_log_odds'].abs()
individual_explanation = individual_explanation.sort_values('abs_shap', ascending=False)
print('该客户raw_probability:', float(pred_test[shap_positions[customer_pos]]), '基准log-odds:', float(shap_bias[customer_pos]))
display(individual_explanation.head(15))
top = shap_importance.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top['feature'], top['mean_abs_shap'], color='#167D8D')
ax.set(xlabel='Mean absolute SHAP (log-odds)', title='LightGBM: locked-model feature explanation')
fig.tight_layout()
fig.savefig(RUN_DIR / 'shap_importance.png', bbox_inches='tight')
plt.show()


## 24. 保存、独立评分接口和回读检查

输出目录包含模型文件、预处理规则、参数/版本/数据指纹、指标、剔除记录、搜索结果、分箱监控规则、
图表、Test分数及`scoring.py`。每次运行新建目录，避免覆盖旧模型。
`scoring.py`从本notebook中的真实函数生成，不依赖notebook里训练好的全局变量；新进程可以只加载文件并评分。
XGBoost编码器使用joblib保存，**只加载自己生成或可信来源的joblib文件**。
原始输入字段缺失会报错，列顺序变化不影响结果，额外的TARGET/ID列不会参与预测。


In [ ]:
def load_bundle(bundle_dir):
    folder = Path(bundle_dir)
    spec = json.loads((folder / 'preprocess_spec.json').read_text(encoding='utf-8'))
    manifest = json.loads((folder / 'model_manifest.json').read_text(encoding='utf-8'))
    if spec['schema_version'] != 1 or spec['model_kind'] != 'lightgbm':
        raise ValueError('不支持的模型或预处理规则版本。')
    model_path = folder / manifest['model_file']
    # 校验整套配套资产，避免模型与别次运行的类别表/编码器混用。
    for filename, expected_hash in manifest['asset_hashes'].items():
        if file_sha256(folder / filename) != expected_hash:
            raise ValueError(f'模型配套文件校验失败：{filename}')
    model = lgb.Booster(model_file=str(model_path))

    enc = None
    return {'spec': spec, 'manifest': manifest, 'model': model, 'encoder': enc}


def predict_new_data(raw, bundle, batch_size=10000):
    if not isinstance(batch_size, int) or batch_size < 1:
        raise ValueError('batch_size必须为正整数。')
    spec = bundle['spec']
    # 即使空表也先核验列结构。
    validate_raw_schema(raw.iloc[:0], spec)
    scores = []
    for start in range(0, len(raw), batch_size):
        X_part = transform_raw(raw.iloc[start:start + batch_size], spec, bundle['encoder'])
        p = bundle['model'].predict(X_part, num_iteration=bundle['manifest']['iterations'])
        scores.append(np.asarray(p, dtype=float))
    prob = np.concatenate(scores) if scores else np.array([], dtype=float)
    if not np.isfinite(prob).all() or ((prob < 0) | (prob > 1)).any():
        raise ValueError('模型产生非法分数。')
    result = pd.DataFrame(index=raw.index)
    if 'SK_ID_CURR' in raw:
        result['SK_ID_CURR'] = raw['SK_ID_CURR'].to_numpy()
    result['raw_probability'] = prob
    result['manual_review_flag_demo'] = (prob >= bundle['manifest']['review_threshold_demo']).astype('int8')
    return result


In [ ]:
model_file = 'model.txt'
final_model.booster_.save_model(str(RUN_DIR / model_file), num_iteration=best_iterations)
if encoder is not None:
    joblib.dump(encoder, RUN_DIR / 'onehot_encoder.joblib')
(RUN_DIR / 'preprocess_spec.json').write_text(json.dumps(preprocess_spec, ensure_ascii=False, indent=2), encoding='utf-8')
monitoring = {'feature_specs': psi_specs, 'score_spec': score_psi_spec,
              'reference': 'Train', 'score_direction': 'higher is riskier', 'alpha': 0.5}
(RUN_DIR / 'monitoring_specs.json').write_text(json.dumps(monitoring, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
manifest = {
    **locked_config, 'model_file': model_file, 'model_sha256': file_sha256(RUN_DIR / model_file),
    'source_file_name': CSV_PATH.name, 'original_rows': original_n, 'used_rows': len(df),
    'n_train': len(y_train), 'n_selection': len(y_selection), 'n_test': len(y_test),
    'test_metrics': get_metrics(y_test, pred_test),
    'test_is_oot': False, 'candidate_reached_cap': bool(best_row['reached_cap']),
    'asset_hashes': {filename: file_sha256(RUN_DIR / filename)
                     for filename in [model_file, 'preprocess_spec.json'] + (['onehot_encoder.joblib'] if encoder is not None else [])}
}
(RUN_DIR / 'model_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

tables = {
    'feature_quality': feature_report, 'dropped_features': basic_drop,
    'feature_mapping': feature_mapping, 'search_results': all_search_result,
    'final_metrics': final_result.reset_index(), 'metric_gaps': gap_table,
    'test_confidence_interval': test_confidence_interval, 'risk_deciles': decile_table,
    'review_demo': review_table.rename_axis('metric').reset_index(),
    'feature_psi': psi_table, 'score_psi': score_psi, 'input_drift': input_drift,
    'feature_importance': feature_importance, 'shap_importance': shap_importance,
    'individual_explanation': individual_explanation
}
for filename, table in tables.items():
    table.to_csv(RUN_DIR / f'{filename}.csv', index=False, encoding='utf-8-sig')
test_predictions = pd.DataFrame({
    'SK_ID_CURR': df.iloc[test_idx]['SK_ID_CURR'].to_numpy(), 'TARGET': y_test.to_numpy(),
    'raw_probability': pred_test, 'manual_review_flag_demo': test_review.astype('int8')
})
test_predictions.to_csv(RUN_DIR / 'test_predictions.csv', index=False, encoding='utf-8-sig')
requirements = '\n'.join(f'{k}=={v}' for k, v in VERSIONS.items() if k != 'python')
(RUN_DIR / 'requirements.txt').write_text(requirements + '\n', encoding='utf-8')

# inspect在Jupyter/IPython中读取刚定义的函数源码，生成不依赖notebook状态的评分模块。
scoring_imports = "from pathlib import Path\nimport hashlib\nimport json\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport lightgbm as lgb\n"
scoring_functions = [file_sha256, safe_div, prepare_features, category_text, validate_raw_schema,
                     transform_model_frame, transform_raw, load_bundle, predict_new_data]
scoring_source = scoring_imports + '\n\n'.join(inspect.getsource(fn) for fn in scoring_functions)
compile(scoring_source, 'scoring.py', 'exec')
(RUN_DIR / 'scoring.py').write_text(scoring_source, encoding='utf-8')
print('已生成评分脚本、模型、输入规则和分析结果。')


In [ ]:
# 真正从刚保存的scoring.py导入，避免只在内存对象上“自测”。
import importlib.util
module_spec = importlib.util.spec_from_file_location('saved_credit_scoring', RUN_DIR / 'scoring.py')
scoring = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(scoring)
saved_bundle = scoring.load_bundle(RUN_DIR)
example_raw = df.iloc[test_idx[:200]].drop(columns='TARGET').copy()
reloaded_scores = scoring.predict_new_data(example_raw, saved_bundle, batch_size=37)
np.testing.assert_allclose(reloaded_scores['raw_probability'], pred_test[:len(example_raw)], atol=2e-6, rtol=1e-5)

# 额外验收：列重排不变、空表可处理、未知类别/缺失可以预测、缺列明确报错。
reordered = scoring.predict_new_data(example_raw.loc[:, example_raw.columns[::-1]], saved_bundle)
np.testing.assert_allclose(reordered['raw_probability'], reloaded_scores['raw_probability'], atol=2e-6, rtol=1e-5)
assert scoring.predict_new_data(example_raw.iloc[:0], saved_bundle).empty
edge = example_raw.iloc[:2].copy()
if cat_cols:
    edge.loc[edge.index[0], cat_cols[0]] = '__NEW_CATEGORY_FOR_TEST__'
for col in preprocess_spec['raw_numeric']:
    edge.loc[edge.index[-1], col] = np.nan
assert np.isfinite(scoring.predict_new_data(edge, saved_bundle)['raw_probability']).all()
try:
    scoring.predict_new_data(example_raw.drop(columns=preprocess_spec['raw_columns'][0]), saved_bundle)
except ValueError:
    pass
else:
    raise AssertionError('缺列测试未触发预期错误。')

verification = {'save_reload_equal': True, 'column_order_invariant': True, 'empty_input': True,
                'missing_unseen_prediction': True, 'missing_column_rejected': True,
                'shap_additivity_checked': True, 'run_mode': RUN_MODE, 'used_rows': len(df)}
(RUN_DIR / 'verification.json').write_text(json.dumps(verification, indent=2), encoding='utf-8')
display(reloaded_scores.head())
display(pd.Series(verification, name='verification'))
print('全部回读与输入边界检查通过。输出目录:', RUN_DIR)


### 新客户评分，不需要重新训练

将整个运行目录保留在一起。在新Python进程中把该目录加入导入路径即可：

```python
from pathlib import Path
import sys
import pandas as pd

bundle_dir = Path('/你的路径/model_outputs/算法名/本次运行目录')
sys.path.insert(0, str(bundle_dir))
from scoring import load_bundle, predict_new_data

bundle = load_bundle(bundle_dir)
new_data = pd.read_csv('/你的路径/new_applications.csv')
new_scores = predict_new_data(new_data, bundle, batch_size=10000)
new_scores.to_csv('new_scores.csv', index=False)
```

模型与预处理文件必须配套使用，部署时复现`requirements.txt`中的版本。
超大CSV可以使用`pd.read_csv(..., chunksize=10000)`逐块读入再评分，避免一次读完。

## 25. 建模结论怎么写、上线前还缺什么

完成后用`final_result`、`gap_table`、`test_confidence_interval`填写区分能力结论；
用`psi_table`与`input_drift`描述当前随机样本分布差异；用SHAP说明模型依赖哪些输入。
不预填任何未实际跑出的效果，也不保证LightGBM或XGBoost一定优于CatBoost/LR。

- **数据口径**：确认申请时点字段可获得，定义违约观察窗、标签成熟期，排除贷后信息和泄露字段。
- **验证口径**：有业务时间字段后做OOT；多笔申请按客户/时间设计切分，测试拒绝样本偏差与样本代表性。
- **开发稳定性**：当前单Selection流程便于对照参考稿；重要上线模型应在开发数据内部做折内预处理的交叉验证/滚动验证，不能把Test并进调参。
- **业务与公平性**：审核敏感属性及代理变量、解释依据、群体差异、数据合法使用和业务审批，不直接把SHAP当作合规拒贷理由。
- **运行监控**：保存字段缺失/未见类别、分布PSI、评分耗时和版本；标签成熟后监控AUC/AR/KS和实际违约率。
- **校准后续**：本文件到此停止，不含Platt、Isotonic、分数转PD、评级分布调整或定价；未经校准的概率不直接当监管PD。


## API核对与资料

业务特征与切分/搜索结构来自本次两份附件；接口细节核对以下官方资料。
资料用于实现与核验，不代表公开数据项目已经满足真实银行投产要求。

- [LightGBM LGBMClassifier](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMClassifier.html)
- [LightGBM early_stopping](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.early_stopping.html)
- [LightGBM 4.6：缺失与类别](https://lightgbm.readthedocs.io/en/v4.6.0/Advanced-Topics.html)
- [LightGBM 4.6 Booster：贡献值与保存](https://lightgbm.readthedocs.io/en/v4.6.0/pythonapi/lightgbm.Booster.html)
